# Phase 2: Exploratory Data Analysis (EDA)
## Fraud Detection System

This notebook performs a comprehensive Exploratory Data Analysis (EDA) on the Credit Card Fraud Detection dataset.
The objective is to understand the dataset structure, missing values, duplicates, target distribution, feature correlations, and potential data leakage prior to preprocessing and modeling.

--- 
## 1. Imports
Importing core data manipulation and visualization libraries.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization styles
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

--- 
## 2. Load Dataset
Loading raw dataset from relative path `../data/raw/creditcard.csv` (or `data/raw/creditcard.csv` depending on the working directory).
The raw dataset is treated as immutable and is loaded into a pandas DataFrame without destructive modifications.

In [1]:
import os

# Determine dataset path relative to repository root or notebooks folder
data_path = os.path.join('..', 'data', 'raw', 'creditcard.csv') if os.path.exists(os.path.join('..', 'data', 'raw', 'creditcard.csv')) else os.path.join('data', 'raw', 'creditcard.csv')

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)
print(f"Successfully loaded raw dataset with shape: {df.shape}")

Loading dataset from: data\raw\creditcard.csv
Successfully loaded raw dataset with shape: (284807, 31)


--- 
## 3. Dataset Overview
Inspecting dataset dimensions, first/last rows, column identifiers, data types, and memory consumption.

In [ ]:
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns\n")

print("--- First 5 Rows ---")
display(df.head())

print("\n--- Last 5 Rows ---")
display(df.tail())

print("\n--- Data Types & Column Summary ---")
df.info()

--- 
## 4. Missing Value Analysis
Checking for null, missing, or NA entries across all dataset columns.

In [ ]:
missing_counts = df.isnull().sum()
missing_percentages = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percentages
})

print(f"Total Missing Values in Dataset: {missing_counts.sum()}")
print("\nMissing Value Summary per Column:")
display(missing_df.sort_values(by='Missing Count', ascending=False).head(10))

--- 
## 5. Duplicate Analysis
Calculating duplicate records in the raw dataset. Note: Duplicates are identified for analytical insight and will not be removed in this phase.

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_pct = (duplicate_count / len(df)) * 100

print(f"Number of Duplicate Rows: {duplicate_count:,}")
print(f"Percentage of Duplicate Rows: {duplicate_pct:.4f}%")

if duplicate_count > 0:
    print("\nSample Duplicate Rows:")
    display(df[df.duplicated(keep=False)].sort_values(by=['Time', 'Amount']).head(6))

--- 
## 6. Target Variable Analysis
Analyzing the class target column `Class` (0 = Legitimate / Non-Fraud, 1 = Fraudulent).
We calculate class counts, percentages, class imbalance ratio, and visualize the severe imbalance.

In [ ]:
target_col = 'Class'
class_counts = df[target_col].value_counts()
class_pcts = df[target_col].value_counts(normalize=True) * 100
imbalance_ratio = class_counts[0] / class_counts[1]

target_summary = pd.DataFrame({
    'Class': ['0 (Legitimate)', '1 (Fraud)'],
    'Count': [class_counts[0], class_counts[1]],
    'Percentage (%)': [class_pcts[0], class_pcts[1]]
})

print("=== Target Class Distribution ===")
display(target_summary)
print(f"\nExact Class Imbalance Ratio: {imbalance_ratio:.2f} : 1")
print(f"(For every 1 fraudulent transaction, there are approximately {imbalance_ratio:.2f} legitimate transactions.)")

# Plot target distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x=['Legitimate (0)', 'Fraud (1)'], y=[class_counts[0], class_counts[1]], ax=ax1, palette=['#2ecc71', '#e74c3c'])
ax1.set_yscale('log')
ax1.set_title('Target Class Counts (Log Scale)')
ax1.set_ylabel('Transaction Count (Log)')
for p in ax1.patches:
    ax1.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', xytext=(0, 5), textcoords='offset points')

colors = ['#2ecc71', '#e74c3c']
ax2.pie([class_counts[0], class_counts[1]], labels=['Legitimate (99.83%)', 'Fraud (0.17%)'],
        autopct='%1.2f%%', startangle=140, colors=colors, explode=(0, 0.2))
ax2.set_title('Target Class Share (%)')

plt.tight_layout()
plt.show()

--- 
## 7. Descriptive Statistics
Examining summary statistics across numerical features, focusing on `Amount`, `Time`, and principal component features (`V1` to `V28`).

In [ ]:
print("=== Summary Statistics for Key Features ===")
display(df[['Time', 'Amount', 'V1', 'V2', 'V3', 'V4', 'V10', 'V14', 'V17']].describe())

print("\n=== Transaction Amount Statistics by Class ===")
amount_by_class = df.groupby('Class')['Amount'].describe()
display(amount_by_class)

print("\n=== Transaction Time (Seconds) Statistics by Class ===")
time_by_class = df.groupby('Class')['Time'].describe()
display(time_by_class)

--- 
## 8. Feature Distribution Analysis
Visualizing feature distributions and comparing characteristics between legitimate and fraudulent transactions.

In [ ]:
# 1. Distribution of Transaction Amount and Time
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.histplot(df['Amount'], bins=50, kde=True, ax=axes[0, 0], color='purple')
axes[0, 0].set_title('Overall Transaction Amount Distribution')
axes[0, 0].set_xlabel('Amount ($)')

sns.boxplot(x='Class', y='Amount', data=df, ax=axes[0, 1], palette=['#2ecc71', '#e74c3c'], showfliers=False)
axes[0, 1].set_title('Transaction Amount by Class (Excluding Outliers)')
axes[0, 1].set_xticklabels(['Legitimate (0)', 'Fraud (1)'])

sns.histplot(df['Time'] / 3600, bins=48, kde=True, ax=axes[1, 0], color='teal')
axes[1, 0].set_title('Transaction Time Distribution (Hours since first transaction)')
axes[1, 0].set_xlabel('Time (Hours)')

sns.kdeplot(data=df, x=df['Time'] / 3600, hue='Class', common_norm=False, ax=axes[1, 1], palette=['#2ecc71', '#e74c3c'])
axes[1, 1].set_title('Transaction Density over Time by Class')
axes[1, 1].set_xlabel('Time (Hours)')

plt.tight_layout()
plt.show()

In [ ]:
# 2. Distribution comparison of key PCA features with strong target separation (V17, V14, V12, V10)
key_pca_features = ['V17', 'V14', 'V12', 'V10', 'V11', 'V4']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feat in enumerate(key_pca_features):
    sns.boxplot(x='Class', y=feat, data=df, ax=axes[i], palette=['#2ecc71', '#e74c3c'], showfliers=False)
    axes[i].set_title(f'{feat} Distribution by Class')
    axes[i].set_xticklabels(['Legitimate', 'Fraud'])

plt.tight_layout()
plt.show()

--- 
## 9. Correlation Analysis
Computing pairwise feature correlations and correlation coefficients with the target column `Class`.

In [ ]:
# Compute correlations with target Class
correlations = df.corr()[target_col].sort_values()

print("=== Top 5 Negative Correlations with Fraud Class ===")
display(correlations.head(5))

print("\n=== Top 5 Positive Correlations with Fraud Class ===")
display(correlations.tail(6).iloc[:-1][::-1])

# Plot Correlation Heatmap
plt.figure(figsize=(14, 10))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, cmap='coolwarm_r', annot=False, linewidths=0.1, cbar=True)
plt.title('Complete Feature Correlation Matrix Heatmap', fontsize=14)
plt.show()

# Plot Bar Chart of Correlations with Class
plt.figure(figsize=(12, 6))
correlations_no_class = correlations.drop(target_col)
colors_corr = ['#e74c3c' if val > 0 else '#3498db' for val in correlations_no_class.values]
correlations_no_class.plot(kind='bar', color=colors_corr)
plt.title('Feature Correlations with Target Class')
plt.ylabel('Pearson Correlation Coefficient')
plt.xlabel('Features')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

--- 
## 10. Potential Data Leakage

### Feature Leakage Review:
1. **Features `V1` to `V28`:** These are numeric features resulting from a Principal Component Analysis (PCA) transformation applied by the original dataset providers to preserve user confidentiality. Because PCA transforms historical transaction attributes present *at transaction time*, these features do not contain future or post-decision information.
2. **`Time`:** Represents elapsed seconds from the first transaction in the dataset. It is known at transaction time and contains no post-decision information.
3. **`Amount`:** Transaction dollar amount. Known at transaction time.
4. **`Class`:** Binary target label (0 or 1). This is the prediction target.

### Preprocessing Leakage Risk:
- **Global Scaling/Resampling Risk:** Performing global feature scaling (e.g., `StandardScaler`) or oversampling (e.g., SMOTE) across the entire dataset prior to train/test splitting introduces data leakage.
- **Mitigation Strategy:** Any feature scaling, normalization, or resampling MUST be fitted strictly on the training set after performing a stratified train/test split.

--- 
## 11. Initial Findings & Modeling Implications

### Summary of Key Findings:
- **Dataset Size:** 284,807 transaction records.
- **Features:** 30 input features (`Time`, `Amount`, `V1` through `V28`) and 1 binary target (`Class`).
- **Class Imbalance:** Extreme class imbalance. Legitimate transactions (0): **284,315 (99.83%)**, Fraudulent transactions (1): **492 (0.17%)**. Imbalance Ratio: **577.88 : 1**.
- **Missing Values:** Zero (0) missing values across all columns.
- **Duplicates:** 1,081 duplicate rows (0.38% of dataset). These should be evaluated carefully during preprocessing.
- **Distribution Observations:**
  - Transaction `Amount` is highly right-skewed with a mean of $88.35, median of $22.00, and max of $25,691.16.
  - Fraudulent transaction amounts have a lower median ($9.25) but display significant variance (max $2,125.87).
  - `V17`, `V14`, `V12`, `V10` show strong negative correlation with fraud, showing clear distributional shifts for fraudulent transactions.
  - `V11` and `V4` show notable positive correlation with fraud.
- **Data Leakage:** No domain-level feature leakage observed. Preprocessing leakage must be prevented by partitioning train/test splits before fitting transformers.

### Implications for Preprocessing & Modeling:
1. **Evaluation Metrics:** Accuracy is meaningless due to 99.83% majority baseline. Evaluation MUST rely on **PR-AUC (Precision-Recall AUC)**, **Recall**, **Precision**, and **F1-Score**.
2. **Resampling / Class Weights:** Class weights (e.g., `class_weight='balanced'`) or sub-sampling/oversampling techniques should be explored to handle the 577.88:1 imbalance.
3. **Feature Scaling:** `Amount` and `Time` require robust scaling (e.g., `RobustScaler` or `StandardScaler`) as PCA features `V1-V28` are already scaled.